# 03 — Baseline Modeling
**Amreen**

Establishing the floor.
- Linear Regression → RUL
- Logistic Regression → binary failure risk

Every model I build next has to beat these numbers.


In [1]:
import sys, os
sys.path.insert(0, '..')
import pandas as pd, numpy as np
from sklearn.model_selection import GroupShuffleSplit
import warnings; warnings.filterwarnings('ignore')

from src.models.baseline import BaselineRULModel, BaselineClassifier
from src.models.evaluate  import evaluate_rul, evaluate_classifier
from src.features.build_features import get_feature_cols

In [2]:
train_feat = pd.read_csv('../data/processed/train_features.csv')
feat_cols  = get_feature_cols(train_feat)
print(f"Features: {len(feat_cols)}, Rows: {len(train_feat)}")

Features: 135, Rows: 19134


## Train/val split — grouped by unit_id to prevent data leakage

In [3]:
# IMPORTANT: split by unit, not by row
# If we split rows randomly, the same engine appears in both train and val
# which would give artificially inflated metrics.
gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, val_idx = next(gss.split(train_feat, groups=train_feat['unit_id']))

trn = train_feat.iloc[train_idx]
val = train_feat.iloc[val_idx]

print(f"Train rows: {len(trn)} | units: {trn['unit_id'].nunique()}")
print(f"Val rows  : {len(val)} | units: {val['unit_id'].nunique()}")

Train rows: 15463 | units: 64
Val rows  : 3671 | units: 16


## RUL Baseline

In [4]:
rul_model = BaselineRULModel()
rul_model.fit(trn[feat_cols], trn['rul'])

val_preds_rul = rul_model.predict(val[feat_cols])
baseline_rul_metrics = evaluate_rul(val['rul'].values, val_preds_rul, label='Linear Regression Baseline')


── Linear Regression Baseline — RUL Regression ──
  RMSE        : 71.56 cycles
  MAE         : 60.15 cycles
  NASA Score  : 81686630  (lower is better)


## Classification Baseline

In [5]:
clf_model = BaselineClassifier()
clf_model.fit(trn[feat_cols], trn['will_fail'])

val_probas = clf_model.predict_proba(val[feat_cols])
baseline_clf_metrics = evaluate_classifier(val['will_fail'].values, val_probas, label='Logistic Regression Baseline')


── Logistic Regression Baseline — Failure Classifier (threshold=0.5) ──
  AUC         : 0.8130
  F1          : 0.4000
  Precision   : 0.2883
  Recall      : 0.6532
  Confusion Matrix:
[[2375  800]
 [ 172  324]]


In [6]:
import os, joblib
os.makedirs('../models/artifacts', exist_ok=True)
rul_model.save('../models/artifacts/baseline_rul.pkl')
clf_model.save('../models/artifacts/baseline_clf.pkl')

print("\nBaseline metrics to beat:")
print(f"  RUL RMSE      : {baseline_rul_metrics['rmse']:.2f}")
print(f"  RUL NASA Score: {baseline_rul_metrics['nasa_score']:.0f}")
print(f"  CLF AUC       : {baseline_clf_metrics['auc']:.4f}")
print(f"  CLF F1        : {baseline_clf_metrics['f1']:.4f}")

  saved to ../models/artifacts/baseline_rul.pkl
  saved to ../models/artifacts/baseline_clf.pkl

Baseline metrics to beat:
  RUL RMSE      : 71.56
  RUL NASA Score: 81686630
  CLF AUC       : 0.8130
  CLF F1        : 0.4000
